In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar `creditsight` sin instalarlo: el código vive en ejemplo/src
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 03 · Preprocesamiento — de las decisiones del EDA a código con pruebas

El EDA terminó con 10 decisiones. Aquí se convierten en **código reutilizable y probado** (en
`src/creditsight/data/`), no en celdas sueltas: el notebook solo orquesta. Hay dos motivos:

1. La limpieza tiene que correr **igual** al entrenar, al validar y, más adelante, al predecir.
2. Una limpieza sin pruebas es una limpieza en la que hay que confiar a ciegas.

Esta etapa produce los datos limpios, la partición **por cliente** y la demostración de por qué esa
partición importa.

In [2]:
import pandas as pd

from creditsight.config import PROCESSED_DIR, TARGET
from creditsight.data.cleaning import STABLE_PER_CUSTOMER, clean, load_raw
from creditsight.data.split import assert_disjoint_customers, split_by_customer

raw = load_raw()
df = clean(raw)
print(f"antes: {raw.shape}  ->  después: {df.shape}  (mismo número de filas: {len(raw) == len(df)})")

antes: (100000, 25)  ->  después: (100000, 26)  (mismo número de filas: True)


## Qué cambió

Comprobamos que cada decisión del EDA se cumple. Esto es una **verificación**, no un gráfico:
si algo no se cumple, este notebook falla.

In [3]:
checks = {
    "no quedan datos personales": not {"ID", "Name", "SSN"} & set(df.columns),
    "sin filas perdidas": len(df) == len(raw),
    "12.500 clientes completos (8 meses)": bool((df.groupby("Customer_ID").size() == 8).all()),
}
for col in STABLE_PER_CUSTOMER:
    checks[f"{col}: un solo valor por cliente"] = bool((df.groupby("Customer_ID")[col].nunique() <= 1).all())
checks["antigüedad crediticia sube 1 mes por mes"] = bool(
    (df.groupby("Customer_ID")["Credit_History_Months"].diff().dropna() == 1).all()
)
checks["todos los numéricos ahora son numéricos"] = bool(
    df[["Age", "Annual_Income", "Num_of_Loan", "Outstanding_Debt", "Monthly_Balance"]].dtypes.map(pd.api.types.is_float_dtype).all()
)
result = pd.Series(checks, name="se cumple")
assert result.all(), result[~result]
result.to_frame()

,se cumple
no quedan datos personales,True
sin filas perdidas,True
12.500 clientes completos (8 meses),True
Age: un solo valor por cliente,True
Occupation: un solo valor por cliente,True
Annual_Income: un solo valor por cliente,True
Num_Bank_Accounts: un solo valor por cliente,True
Num_Credit_Card: un solo valor por cliente,True
Interest_Rate: un solo valor por cliente,True
Num_of_Loan: un solo valor por cliente,True


## Lo que queda vacío (y por qué está bien)

La limpieza no inventa datos: solo repara lo que se puede deducir con seguridad (valores del propio
cliente, reglas exactas). Lo que sigue vacío se **imputa dentro del pipeline** (notebook 04), con
estadísticas calculadas **solo con el entrenamiento**.

In [4]:
missing = df.isna().mean().loc[lambda s: s > 0].sort_values(ascending=False).rename("% vacío")
missing.to_frame().style.format("{:.1%}")

,% vacío
Type_of_Loan,11.4%
Amount_invested_monthly,8.8%
Num_of_Delayed_Payment,8.4%
Payment_Behaviour,7.6%
Num_Credit_Inquiries,3.6%
Total_EMI_per_month,3.1%
Changed_Credit_Limit,2.1%
Monthly_Balance,1.2%


## La partición por cliente

Cada cliente aparece 8 veces. Probamos las dos formas de partir y comparamos cuánto *parece* saber el
modelo en cada una. Usamos un modelo rápido y los mismos datos; **solo cambia cómo se parte**.

In [5]:
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from creditsight.models.train import build_pipeline, encode_target, expected_cost

def holdout_score(train_df, test_df):
    model = build_pipeline(LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1), scale=False)
    model.fit(train_df.drop(columns=[TARGET]), encode_target(train_df[TARGET]))
    y_true = encode_target(test_df[TARGET])
    y_pred = model.predict(test_df.drop(columns=[TARGET]))
    return f1_score(y_true, y_pred, average="macro"), expected_cost(y_true, y_pred)

# A) por fila: los meses de un mismo cliente caen a ambos lados
row_train, row_test = train_test_split(df, test_size=0.2, random_state=42)
shared = len(set(row_train.Customer_ID) & set(row_test.Customer_ID))

# B) por cliente: cada cliente completo va a un solo lado
cust_train, cust_test = split_by_customer(df, test_size=0.2)
assert_disjoint_customers(cust_train, cust_test)

f1_row, cost_row = holdout_score(row_train, row_test)
f1_cust, cost_cust = holdout_score(cust_train, cust_test)
pd.DataFrame(
    {"clientes en ambos lados": [shared, 0], "F1 macro": [f1_row, f1_cust], "costo esperado": [cost_row, cost_cust]},
    index=["partición por fila (mal)", "partición por cliente (bien)"],
).round(3)

,clientes en ambos lados,F1 macro,costo esperado
partición por fila (mal),10449,0.734,0.65
partición por cliente (bien),0,0.695,0.73


**Lectura.** Con la partición por fila el modelo parece **mejor**: unos 4 puntos más de F1 y
menos costo esperado. Es una ventaja **falsa**, y se ve incluso con un modelo rápido y moderado: ha visto a casi todos los clientes de la prueba en el entrenamiento, con otros
meses, y reconoce al cliente en lugar de aprender a predecir. En producción llegarán clientes
**nuevos**, y ahí el rendimiento real es el de la fila de abajo.

Esta es una **fuga de información** sin un solo error de código: el modelo funciona, el pipeline
funciona, las métricas se ven bien y todo está mal. Por eso la partición es una decisión de diseño,
no un detalle.

> Regla del curso: **la partición respeta la unidad de análisis** y se verifica con una prueba
> automática (`tests/test_split.py`), no a ojo.

## Guardar los datos de la etapa

Partimos **una sola vez**. El conjunto de prueba se guarda y **no se vuelve a tocar** hasta la
etapa 7 (validación final). Todo lo demás se decide solo con entrenamiento.

In [6]:
import joblib

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train, test = split_by_customer(df, test_size=0.2)
joblib.dump(train, PROCESSED_DIR / "train.joblib")
joblib.dump(test, PROCESSED_DIR / "test.joblib")
print(f"entrenamiento: {len(train):,} filas, {train.Customer_ID.nunique():,} clientes")
print(f"prueba:        {len(test):,} filas, {test.Customer_ID.nunique():,} clientes  (cerrada hasta la etapa 7)")
print("distribución del objetivo (entrenamiento):", train[TARGET].value_counts(normalize=True).round(3).to_dict())
print("distribución del objetivo (prueba):       ", test[TARGET].value_counts(normalize=True).round(3).to_dict())

entrenamiento: 80,000 filas, 10,000 clientes
prueba:        20,000 filas, 2,500 clientes  (cerrada hasta la etapa 7)
distribución del objetivo (entrenamiento): {'Standard': 0.532, 'Poor': 0.291, 'Good': 0.177}
distribución del objetivo (prueba):        {'Standard': 0.529, 'Poor': 0.286, 'Good': 0.185}


## Las pruebas automáticas

La limpieza y la partición tienen pruebas en `ejemplo/tests/` (`test_cleaning.py`, `test_split.py`).
Para ejecutarlas desde la raíz del repo:

```bash
pytest ejemplo/tests -q
```

Cada decisión del EDA tiene al menos una prueba, y la regla más importante tiene la suya: *ningún
cliente aparece en dos particiones*.

**Entregable de la etapa:** `train.joblib` y `test.joblib` (en `ejemplo/data_processed/`, no se
versionan: se regeneran con este notebook), más el código en `src/creditsight/data/`.

**Siguiente:** `04_features.ipynb`.